In [5]:
import pandas as pd

In [6]:
train = pd.read_csv('train.csv')
train.head()

,id,trip_duration_min,distance_km,battery_level_start,temperature_c,wind_speed,demand_index,distance_km_noisy,city_zone,scooter_model,is_weekend,rental_price,avg_price_last_week
0,0,25.599707,7.441135,78.519717,6.128784,8.680982,0.861114,11.102968,center,C,0,17.256713,16.734476
1,1,57.289287,8.770495,85.957721,NaN,8.812507,0.078338,9.426186,park,A,1,12.305829,12.411778
2,2,45.259667,6.147492,31.714901,26.099837,3.501865,0.072575,6.285575,center,A,0,11.246925,10.948370
3,3,37.926217,7.740660,86.634377,11.560209,0.832577,0.850318,8.455020,park,B,1,18.671568,18.711083
4,4,13.581025,3.846835,63.223723,15.542815,7.245579,0.212850,7.641726,suburb,B,0,5.406407,5.369427


In [8]:
test = pd.read_csv('test.csv')
test.head()

,id,trip_duration_min,distance_km,battery_level_start,temperature_c,wind_speed,demand_index,distance_km_noisy,city_zone,scooter_model,is_weekend,avg_price_last_week
0,0,8.836442,1.877240,47.638799,27.553230,1.263080,0.938387,1.491197,center,C,1,7.288184
1,1,16.117843,3.137195,56.240545,26.415378,3.716338,0.309659,4.561956,suburb,C,0,9.258660
2,2,48.670206,15.191036,92.183623,15.224930,5.898825,0.818875,14.002809,park,C,1,7.601810
3,3,54.276188,9.917750,75.053843,5.555536,7.039429,0.654849,8.815836,park,A,0,7.024768
4,4,20.555200,4.671866,24.939976,20.907682,0.782399,0.280185,3.988865,park,A,0,7.800213


In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor

# Load data

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

test_ids = test["id"]

# Fill missing

for col in test.columns:

    if train[col].dtype != "object":

        median = train[col].median()

        train[col] = train[col].fillna(median)
        test[col] = test[col].fillna(median)

# Encode categorical

cat_cols = train.select_dtypes(
    include="object"
).columns

encoder = OrdinalEncoder()

train[cat_cols] = encoder.fit_transform(
    train[cat_cols]
)

test[cat_cols] = encoder.transform(
    test[cat_cols]
)

# Features

X = train.drop(
    ["rental_price"],
    axis=1
)

y = train["rental_price"]

# Train

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

model.fit(X, y)

# Predict

preds = model.predict(test)

submission = pd.DataFrame({
    "id": test_ids,
    "rental_price": preds
})

submission.to_csv(
    "submission.csv",
    index=False
)